# NPUWattch hands-on — 4. Bringing your own data

Part 4 of the basics lists five ways to put your own data into a run. This
notebook does the three you are most likely to need:

1. **A user component**: a block NPUWattch has no model for, costed with your
   own numbers.
2. **A compound with its projection**: the same block built from primitives
   NPUWattch already models.
3. **Your own DRAM table.**

All three are done on files you already have from notebooks 1 to 3.


In [1]:
import sys; sys.path.insert(0, ".")
from tutorial_helpers import *
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)
print("npuwattch:", npuwattch_binary())

npuwattch: /home/KNUEEhdd1/ys_knu/kmk/miniforge3/envs/npuwattch/bin/npuwattch


## 1. Add a block of your own

Start from the native files that notebook 1 wrote (`tpu_like_fp32/native/`).
If they are missing, the next cell writes them again.

We add a post-processing unit to the TPU-like design: 128 copies of the NVDLA
SDP from notebook 3, each with its measured area, leakage, and per-action
energy. We tell NPUWattch about it in two places:

- in the description, a `user_components:` block with the entry and one
  component of class `nvdla_sdp`;
- in the activity table, one row that charges a `process` action per output
  element of the matmul (1024 × 1024 = 1 048 576 elements).

The cell edits copies of the two files, so the originals stay as they were.

In [2]:
import yaml, csv, pathlib
ex = example("pytorchsim/tpu_like_fp32")
if not (ex / "native" / "description.yaml").exists():
    _ = run(ex, "-o", "native/", log_name="console_native.txt")

SDP = {
    "reference": {"node": "7nm", "corner": "TT", "voltage_V": 0.75, "temperature_C": 25.0, "clock_MHz": 800.0},
    "design_class": "compute",
    "area_um2": 1942.0,
    "leak_power_mW": 0.01159,
    "actions": {"idle": {"energy_pJ": 9.09}, "process": {"energy_pJ": 15.65}},
}

desc = yaml.safe_load((ex / "native" / "description.yaml").read_text())
desc["npuwattch"]["user_components"] = {"nvdla_sdp": SDP}
desc["npuwattch"]["components"].append({"name": "post.sdp", "class": "nvdla_sdp", "count": 128})
(ex / "native" / "description_sdp.yaml").write_text(yaml.safe_dump(desc, sort_keys=False))

rows = list(csv.reader((ex / "native" / "activity.csv").read_text().splitlines()))
header, body, meta = rows[0], [r for r in rows[1:] if r[3] != "__meta__"], [r for r in rows[1:] if r[3] == "__meta__"]
window, start, end = body[0][0], body[0][1], body[0][2]          # same window as the kernel
body.append([window, start, end, "post.sdp", "op", "process", "1048576"])
with open(ex / "native" / "activity_sdp.csv", "w", newline="") as f:
    csv.writer(f).writerows([header] + body + meta)

print("added component:", desc["npuwattch"]["components"][-1])
print("added row:      ", body[-1])

added component: {'name': 'post.sdp', 'class': 'nvdla_sdp', 'count': 128}
added row:       ['0', '0', '52021', 'post.sdp', 'op', 'process', '1048576']


In [3]:
text_sdp = run_npuwattch(["-d", "native/description_sdp.yaml", "-l", "native/activity_sdp.csv", "--report", "out_sdp/"], cwd=ex)
rep_sdp = report(ex, "out_sdp/report.json")
df = components_df(rep_sdp)
display(df[df["component"] == "post.sdp"])
show_totals(text_sdp)

,component,class,model,instances,energy,share %,area
7,post.sdp,nvdla_sdp,user,128,16.5 µJ,0.3,1.94e+03


total energy = 6.149e+09 pJ (dyn 6.134e+09 + leak 1.476e+07); avg power = 1.111e+05 mW; exec = 5.534e-05 s
calibrated primitives available: fpadd, fpmul, fpmac, intadd, intmul, intmac, fpsfu, mxfpmac, fifo, regfile, simplemux, crossbar, fattree, foldedclos, sram


Expected: `post.sdp` appears with `user` in the `model` column, 128
instances, an area of 128 × 1942 µm² ≈ 0.25 mm², and a dynamic energy of
1 048 576 × 15.65 pJ ≈ 16.4 µJ. Everything else is unchanged.

Now run the same description with the original activity table, which has no
row for the SDP. The block keeps its area and leakage and loses its dynamic
energy, which is what an unused block should do.

In [4]:
text_norow = run_npuwattch(["-d", "native/description_sdp.yaml", "-l", "native/activity.csv", "--report", "out_sdp_norow/"], cwd=ex)
df = components_df(report(ex, "out_sdp_norow/report.json"))
display(df[df["component"] == "post.sdp"])

,component,class,model,instances,energy,share %,area
14,post.sdp,nvdla_sdp,user,128,82.1 nJ,0.0,1.94e+03


Where do the numbers in a user component come from? From whatever source you
trust: your own RTL through synthesis and place and route, a vendor datasheet,
or a published measurement. State the node they belong to in `reference:`. If
the run uses a different node, NPUWattch uses your values as given and tells
you about the difference.

## 2. The same block as a compound: `compound_components.yaml` + `projection.yaml`

A user component is the right choice when you have measured the whole block.
When the block is made of pieces NPUWattch already models, you can describe
it as a **compound** instead, and the trained models cost every piece at your
node. Two files do this, and both harnesses read them from the run folder:

| File | What it says | The fields |
| --- | --- | --- |
| `compound_components.yaml` | What the block **is**: a class name, and the elements (primitives) it is made of | per element: `primitive`, `config` (the primitive's attributes), `count` |
| `projection.yaml` | What the block **does** when the simulator counts an event: which elements work, in which stimulus mode | per action: `count_from` (the counter and a scale), `elements: {element: mode}` |

Let's rebuild the NVDLA SDP from notebook 3 this way. The SDP has eight
lanes; each lane multiplies an int32 partial sum by an int16 scale and adds a
bias. So the compound is 8 × `intmul` + 8 × `intadd`. Its one action is
`write`: every output element that leaves the chip passes through one lane.
The cell below copies the NVDLA-like design into a work folder and makes
three changes:

1. `arch.yaml`: the `sdp` component gets `class: sdp_logic` instead of the
   `user_component` attribute.
2. `compound_components.yaml`: the `sdp_logic` compound is added.
3. `projection.yaml`: the `write` action of `sdp_logic` is added.

`stats_map.yaml` stays almost the same: each DRAM write (one 64-bit word,
eight int8 elements) fans out to eight `write` events on the SDP.

In [5]:
import shutil
src = example("timeloop/nvdla_like")
work = pathlib.Path("work") / "nvdla_sdp_compound"
if work.exists():
    shutil.rmtree(work)
shutil.copytree(src, work, ignore=shutil.ignore_patterns("resnet50", "llama3_8b"))

# 1. arch.yaml: the SDP is now a compound class, not a user component
arch = (work / "arch.yaml").read_text()
arch = arch.replace('class: dummy_storage\n    attributes:\n      user_component: "nvdla_sdp"\n',
                    'class: sdp_logic\n    attributes:\n')
(work / "arch.yaml").write_text(arch)

# 2. the compound: what the SDP is made of
COMPOUND = '''
sdp_logic:                            # the NVDLA post-processing unit, 8 lanes
  default_mode: gated                 # elements an action does not name are clock-gated
  elements:
    mul:                              # int32 partial sum x int16 scale
      primitive: intmul
      config: {data_width_a: 32, data_width_b: 16, data_width_out: 48, pipeline_stages: 5}
      count: 8
    add:                              # + int48 bias
      primitive: intadd
      config: {data_width_a: 48, data_width_b: 48, data_width_out: 48, pipeline_stages: 2}
      count: 8
'''
# 3. the projection: what one event does to the elements
PROJECTION = '''
  sdp_logic:
    write:                            # one output element leaving the chip
      count_from: {stat: writes, scale: 1}
      elements: {mul: random, add: random}
'''
with open(work / "compound_components.yaml", "a") as f: f.write(COMPOUND)
with open(work / "projection.yaml", "a") as f: f.write(PROJECTION)
(work / "stats_map.yaml").write_text("levels:\n  DRAM:\n    read: DRAM\n    write: [DRAM, {sdp: 8}]\n")
print(COMPOUND); print(PROJECTION); print((work / "stats_map.yaml").read_text())


sdp_logic:                            # the NVDLA post-processing unit, 8 lanes
  default_mode: gated                 # elements an action does not name are clock-gated
  elements:
    mul:                              # int32 partial sum x int16 scale
      primitive: intmul
      config: {data_width_a: 32, data_width_b: 16, data_width_out: 48, pipeline_stages: 5}
      count: 8
    add:                              # + int48 bias
      primitive: intadd
      config: {data_width_a: 48, data_width_b: 48, data_width_out: 48, pipeline_stages: 2}
      count: 8


  sdp_logic:
    write:                            # one output element leaving the chip
      count_from: {stat: writes, scale: 1}
      elements: {mul: random, add: random}

levels:
  DRAM:
    read: DRAM
    write: [DRAM, {sdp: 8}]



In [6]:
text_c = run_npuwattch(["--harness", "timeloop", "--arch-yaml", "arch.yaml",
                        "--stats", str(src / "resnet50" / "stats"), "--stats-map", "stats_map.yaml",
                        "--node", "7nm", "--clock-mhz", "1000", "--report", "out/"], cwd=work)
show_messages(text_c, kinds=("WARNING",))
for level, code, line in messages(text_c):
    if level == "INFO" and "sdp" in line:
        print(line[:150])
rep_c = report(work)
df_c = components_df(rep_c)
display(df_c[df_c["component"].str.contains("sdp")])

(no WARNING lines)
INFO (NW-7110): system_top_level.nvdla.sdp: class 'sdp_logic' is the compound component 'sdp_logic' (mul, add).
INFO (NW-2116): User component 'nvdla_sdp' (user_components.yaml) is in the library, but the run does not use it.
INFO (NW-7222): Stats level 'DRAM' charges more than one component: read → system_top_level.DRAM; write → system_top_level.DRAM, system_top_level.nvd


,component,class,model,instances,energy,share %,area
6,system_top_level.nvdla.sdp.mul,intmul,cal,8,7.13 µJ,0.6,366
7,system_top_level.nvdla.sdp.add,intadd,cal,8,1.49 µJ,0.1,77.8


The SDP now appears as two rows, `sdp.mul` and `sdp.add`, with `cal` in the
`model` column: their cost came from the trained `intmul` and `intadd` models
at 7 nm, not from a number you typed. Side by side with the measured user
component from notebook 3:

In [7]:
rep_u = report(example("timeloop/nvdla_like/resnet50"))
sdp_u = next(c for c in rep_u["components"] if c["name"].endswith(".sdp"))
sdp_c = [c for c in rep_c["components"] if ".sdp." in c["name"]]
pd.DataFrame([
    {"SDP as": "user component (measured RTL)", "model": "user", "energy": sdp_u["energy_str"], "share %": round(sdp_u["energy_pct"], 1), "area um2": round(sdp_u["area_um2"])},
    {"SDP as": "compound (intmul + intadd)", "model": "cal",
     "energy": f"{sum(c['energy_pJ'] for c in sdp_c) / 1e6:.3g} µJ", "share %": round(sum(c["energy_pct"] for c in sdp_c), 1),
     "area um2": round(sum(c["area_um2"] for c in sdp_c))},
])

,SDP as,model,energy,share %,area um2
0,user component (measured RTL),user,24.1 µJ,2.0,1942
1,compound (intmul + intadd),cal,8.63 µJ,0.7,3553


Both are a small share of the total, and the two differ by about a factor of
three, because they model different things: the user component is the full
SDP RTL (bias, batch norm, ReLU, and requantization) measured once, while the
compound is the multiply-and-add datapath built from calibrated pieces. Use a
user component when you have measured the block. Use a compound when you want
the block to follow the node and the parameters you give it, and when you want
each piece in the breakdown.

The same two files are what the PyTorchSim harness runs on (notebook 1,
section 4): there, the compounds describe the whole machine, and the
projection names the counters of the TOGSim log and the gem5 statistics.

## 3. Your own DRAM table

The DRAM cost comes from a small table: energy per transferred bit (split
into DRAM, I/O, and PHY), energy per row activation, and an optional refresh
term. NPUWattch ships tables for LPDDR4, LPDDR, DDR3, GDDR5, HBM2, and HMC;
the Timeloop harness picks one by the Accelergy `type` of the DRAM. Here is
the HBM2 table:

In [8]:
import npuwattch, pathlib
tables = pathlib.Path(npuwattch.__file__).parent / "energy" / "dram_tables"
print(sorted(p.name for p in tables.glob("*.yml")))
print()
show_file(tables / "hbm2.yml")

['ddr3.yml', 'gddr5.yml', 'hbm2.yml', 'hmc.yml', 'lpddr.yml', 'lpddr4.yml']

# HBM2 off-chip DRAM energy constants, in pJ.
#
# This is the default DRAM energy table of NPUWattch. All default `hbm`
# constants come from this file. The values are the same as the hbm2.yml of
# the PyTorchSim authors.
#
# The values are literature constants. No public HBM power tool or vendor IDD
# datasheet is available. If you change a value, update this citation also.
# Source:
#   M. O'Connor, N. Chatterjee, D. Lee, J. Wilson, A. Agrawal, S. W. Keckler,
#   and W. J. Dally, "Fine-Grained DRAM: Energy-Efficient DRAM for Extreme
#   Bandwidth Systems," in Proc. 50th Annual IEEE/ACM International Symposium
#   on Microarchitecture (MICRO-50), 2017, Table 3 (HBM2 column).
#   DOI: 10.1145/3123939.3124545
#
# row_activation_pj: 909
#     Row activation: precharge and activate of one 1 KB row.
#     Source: Table 3, "Row activation".
# transfer_pj_per_bit: 3.48 in total
#     Access energy for each bit, from

`--energy-table` replaces the table for a run. The Eyeriss-like design in
notebook 2 declares `type: LPDDR4` (8 pJ/bit). Run it with the HBM2 table
(3.48 pJ/bit) and compare. Only the DRAM line changes.

In [9]:
ex1 = example("timeloop/eyeriss_like/alexnet")
text_lpddr4 = run(ex1, log_name="console_lpddr4.txt")
rep_lpddr4 = report(ex1)
text_hbm2 = run(ex1, "--energy-table", str(tables / "hbm2.yml"), "--report", "out_hbm2/", log_name="console_hbm2.txt")
rep_hbm2 = report(ex1, "out_hbm2/report.json")
pd.DataFrame([summary_row(rep_lpddr4, "LPDDR4 (shipped table)"), summary_row(rep_hbm2, "HBM2 (--energy-table)")]).drop(columns="total_pJ")

,design / workload,total energy,DRAM share %,area mm²,avg power,pJ / MAC
0,LPDDR4 (shipped table),1.16e+04 µJ,39.8,0.459,1.79 W,16.18
1,HBM2 (--energy-table),8.96e+03 µJ,22.4,0.459,1.39 W,12.54


In [10]:
show_messages(text_hbm2, kinds=("INFO",))

INFO (NW-1026): Harness mode: timeloop
INFO (NW-1301): NPUWattch builds the description database from ../arch.yaml.
INFO (NW-1303): The description database has 6 components with 674 instances in total.
INFO (NW-1001): Instance hierarchy (declared in the Accelergy description):
INFO (NW-7401): system_top_level.DRAM: 3.48 pJ/bit from --energy-table 'HBM2' (hbm2.yml: dram 1.51 + io 1.17 + phy 0.8).
INFO (NW-7301): system_top_level.DRAM (hbm): NPUWattch ignores the Accelergy attribute(s) datawidth.
INFO (NW-7306): system_top_level.eyeriss.shared_glb (sram): depth 16384 is the total of 32 banks, so mem_depth_per_bank is 512 (128 KB total).
INFO (NW-7307): system_top_level.eyeriss.shared_glb (sram): no port count is declared, so NPUWattch uses one shared read-write port.
INFO (NW-7308): system_top_level.eyeriss.shared_glb (sram): bandwidth 16 words/cycle (8 words per access) gives up to 2 bank accesses per cycle.
INFO (NW-7301): system_top_level.eyeriss.shared_glb (sram): NPUWattch ignores 

To use your own numbers, write a file in the same format (the per-bit
terms are required; the activation and refresh terms are optional for
Timeloop) and pass it with `--energy-table`. In a native description, the
same values can be written directly as attributes of an `hbm` component.

## 4. The other two ways

- **The native files for any simulator**: notebook 1, section 5 showed the two
  files. Write them from your simulator's statistics and run `npuwattch -d
  description.yaml -l activity.csv`.
- **Retraining**: the training scripts ship with the package and accept a
  dataset directory in our CSV layout. See Part 4, §1.5 of the basics.

## Messages you will see

| What you see | What it means | What to do |
| --- | --- | --- |
| `INFO (NW-2116): User component 'X' (...) is in the library, but the run does not use it.` | An entry in the library that no component refers to | Nothing. Remove the entry, or add `--suppress NW-2116`, if you want a quiet run |
| `WARNING (NW-8302): ...: User component 'X': the values are for <node>, but the run is at <node>.` | Your numbers belong to another node | Nothing is wrong; the values are used as given. We are preparing to support scaling user components across nodes in a future release |
| `WARNING (NW-6509): <kernel>: the run declares the DRAM energy table ..., but NPUWattch charges ... from --energy-table.` (PyTorchSim) | The table you passed is not the one the simulator ran with | Pass the run's own table, or keep yours if that is the point |

That is the end of the hands-on. The basics folder (`tutorial/npuwattch_basic/`)
has the full description of every file and option used here.
